In [2]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

In [15]:
from src.language_models.model import RNNModel as lstm
import torch
from wm_tests.utils import WMTestDataset, collate_fn
from src.language_models.dictionary_corpus import Dictionary
from torch.utils.data import DataLoader
import torch.nn.functional as F
from src.language_models.utils import move_to_device, repackage_hidden

In [16]:
device = torch.device('cpu')
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
model = lstm("LSTM", len(dictionary), 650, 650, 2, 0.2, False).to(device)
checkpoint = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/epoch_40.pt', map_location=device)
batch_size=230

In [17]:
cat_s3_repeat_marker = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce3_repeat_markers.txt'
cat_s3_repeat = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce3_repeat.txt'
cat_s3_repeat_dataset = WMTestDataset(cat_s3_repeat,cat_s3_repeat_marker, dictionary)
cat_s3_repeat_dataloader = DataLoader(cat_s3_repeat_dataset, batch_size=batch_size, collate_fn=collate_fn)

In [19]:
def eval(model, dataloader, batch_size):
    all_repeat_surprisals = {}
    model.eval()
    hidden = move_to_device(model.init_hidden(batch_size), device)
    # Forward pass with hidden state update word by word
    with torch.no_grad():
        for batch in dataloader:
            encoded_sentence = batch["encoded_sentence"]
            condition = batch["condition"]
            marker = batch["marker"]
            batch_size, seq_len = encoded_sentence.shape
            input_seq = encoded_sentence[:, :-1].transpose(0, 1)  # (seq_len-1, batch_size)
            target_seq = encoded_sentence[:, 1:].transpose(0, 1)
            
            
            output, hidden = model(input_seq,hidden)
            hidden = repackage_hidden(hidden)
            
            log_probs = F.log_softmax(output, dim=-1)  #
            
            nll_loss = F.nll_loss(
                        log_probs.reshape(-1, log_probs.size(-1)),  # ( (seq_len-1)*batch_size, vocab_size )
                        target_seq.reshape(-1),                         # ((seq_len-1)*batch_size)
                        reduction='none'
                    )
                    
                    # Reshape back to (seq_len-1, batch_size)
            nll_loss = nll_loss.view(seq_len - 1, batch_size).transpose(0, 1)  # (batch_size, seq_len-1)        
            mask_list1 = (marker[:, 1:] == 1)  # remove first token since nll_loss aligns with shifted target
            mask_list2 = (marker[:, 1:] == 3)
            
            # Extract surprisal for each list and reshape
            # Number of tokens in each list should be condition[0][0]*2 (including punctuation)
            list_len = condition[0][0] * 2
            
            surprisal_list1 = nll_loss[mask_list1].view(batch_size, list_len)
            surprisal_list2 = nll_loss[mask_list2].view(batch_size, list_len)
            
            # Select repeated word indices (odd positions assuming repeats are at odd indices)
            word_indices = torch.arange(0, condition[0][0]*2, step=2)  # e.g., 1, 3, 5, ...
            word_indices = word_indices[1:]#get rid of first word of the list
  
            surprisal1_repeats = surprisal_list1[:, word_indices]
            surprisal2_repeats = surprisal_list2[:, word_indices]
            
            # Compute repeat surprisal ratio as percentage
            repeat_surprisal = (surprisal2_repeats / surprisal1_repeats) * 100
            all_repeat_surprisals[f'list len : {condition[0][0]}, prompt len : {condition[0][1]}']=repeat_surprisal
            
    return all_repeat_surprisals
        

In [21]:
repeat_surprisal = eval(model, cat_s3_repeat_dataloader, batch_size)